In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('data/consumer.csv')
df = df.rename(columns = {'Education':'education', 'Purchased':'purchased', 'Review':'review'})
df['review'] = df['review'].str.lower()
df.columns, df.sample(5)

(Index(['Age', 'Gender', 'review', 'education', 'purchased'], dtype='object'),
     Age  Gender   review education purchased
 37   26    Male     poor        HS        No
 39   31    Male  average        UG       Yes
 23   48    Male     good        PG       Yes
 24   34  Female  average        UG       Yes
 34   21  Female     poor        HS        No)

# Ordinal Encoding for Ordinal Categorical Columns

In [ ]:
df = df[['review', 'education', 'purchased']]

X = df[['review', 'education']]
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)
oe = OrdinalEncoder(dtype= int, categories = [['poor', "average", "good"], ['HS', 'UG', 'PG']])

oe.fit(X_train)

X_train = pd.DataFrame(oe.transform(X_train), columns=X.columns)
X_test = pd.DataFrame(oe.transform(X_test), columns= X.columns)

X_train.head(5)


To check how the order is given (lower to higher)

In [48]:
oe.categories_

[array(['poor', 'average', 'good'], dtype=object),
 array(['HS', 'UG', 'PG'], dtype=object)]

# Label Encoding for Target Categorical Column

In [107]:
LE = LabelEncoder()

LE.fit(y_train)

y_train = LE.transform(y_train)
y_test = LE.transform(y_test)

y_train = pd.DataFrame(y_train, columns= [y.name])
y_test = pd.DataFrame(y_test, columns= [y.name])

y_train.head()

,purchased
0,1
1,1
2,0
3,0
4,0


In [68]:
LE.classes_

array(['No', 'Yes'], dtype=object)

In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(random_state=0)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy_score(y_test, predictions)

C:\Users\Keval\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1.0

# One Hot Encoding using get_dummies()

In [8]:
df = pd.get_dummies(df, columns=['Gender'], drop_first=True, prefix="", prefix_sep="", dtype= np.int32) # drop_first for multicollinearity
# pd.get_dummies(df, columns=['Gender']) # without drop_first

In [9]:
df.sample(5)

,Age,review,education,purchased,Male
16,23,poor,HS,No,0
21,30,average,UG,Yes,1
9,24,poor,HS,No,1
49,27,poor,HS,No,1
10,29,good,PG,Yes,0


___

# Multicollinearity 
when two or more input columns are highly co-related and that shound not be happening in the machine learning model

---

# One Hot Encoding for Nominal Categorical data

In [99]:
OHE = OneHotEncoder(dtype = np.int32, sparse_output = False) # use drop = 'first' if want to remove the first column

X = df[['review', 'education']]
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [102]:
X_train.sample(5)

,review,education
32,good,PG
49,poor,HS
24,average,UG
33,average,UG
12,poor,HS


In [106]:
X_train_new = pd.DataFrame(OHE.fit_transform(X_train))
X_test_new = pd.DataFrame(OHE.transform(X_test))

# OHE removes the indexes so we have to give indexes
X_train_new.index = X_train.index
X_test_new.index = X_test.index

# remove the Categorical columns from the X_train
num_X_train = X_train.drop(['review', 'education'], axis = 1)
num_X_test = X_test.drop(['review', 'education'], axis = 1)

# jointing the encoded columns 
OH_X_train = pd.concat([num_X_train, X_train_new], axis = 1)
OH_X_test = pd.concat([num_X_test, X_test_new], axis = 1)

C:\Users\Keval\AppData\Local\Temp\ipykernel_6180\2710500364.py:13: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  OH_X_train = pd.concat([num_X_train, X_train_new], axis = 1)
C:\Users\Keval\AppData\Local\Temp\ipykernel_6180\2710500364.py:14: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  OH_X_test = pd.concat([num_X_test, X_test_new], axis = 1)


In [108]:
model = LogisticRegression(random_state=0)

model.fit(OH_X_train, y_train)

pred = model.predict(OH_X_test)

accuracy_score(y_test, pred)

C:\Users\Keval\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1.0